<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# 🔤 Configure LiteLLM: Your Universal AI Gateway

Welcome to the second step of building your **AI Research Assistant**! In notebook 00, you validated that all platform services are running. Now it's time to **configure your AI models** so they're accessible through a unified API.

## 🎯 What You'll Learn

In this notebook, you'll:
- ✅ **Register your local LLM** (GPT-OSS 20B) in LiteLLM
- ✅ **Register your embedding model** (nomic-embed-text-v1.5) in LiteLLM
- ✅ **Create a unified OpenAI-compatible API** for both models
- ✅ **Test end-to-end** to ensure everything works

## 💡 Why This Matters

Without LiteLLM, you'd need to manage multiple APIs:
- ❌ One API for your chat model (`tkt-tensorrt-llm`)
- ❌ Another API for embeddings (`tkt-text-embeddings`)
- ❌ Different authentication schemes
- ❌ Inconsistent response formats

**With LiteLLM**, you get:
- ✅ **Single endpoint** for all AI capabilities
- ✅ **OpenAI-compatible API** (works with LangChain, CrewAI, etc.)
- ✅ **Cost tracking** across all models
- ✅ **Load balancing & fallbacks** out of the box
- ✅ **Easy model switching** without code changes

## 🏗️ What We're Building

```mermaid
graph LR
    A[🤖 Your Application<br/>LangChain/CrewAI] --> B[🔤 LiteLLM Gateway<br/>Single API]
    B --> C[💬 Chat Model<br/>tkt-tensorrt-llm<br/>GPT-OSS 20B]
    B --> D[🧮 Embeddings<br/>tkt-text-embeddings<br/>nomic-embed-text-v1.5]
    
    style A fill:#e1f5ff
    style B fill:#fff4e6
    style C fill:#f3e5f5
    style D fill:#e8f5e9
```

### 🔄 How Registration Works

1. **Deploy Models** - Your models are already running on Kubernetes
   - `tkt-tensorrt-llm` → Chat completions (GPT-OSS 20B)
   - `tkt-text-embeddings` → Vector embeddings (nomic-embed-text-v1.5)

2. **Register in LiteLLM** - Tell LiteLLM where to find your models
   - Model name (e.g., `gpt-oss`, `nomic-embed`)
   - Service endpoint (e.g., `https://gptoss20.domain.com`)
   - Model type (chat vs. embedding)

3. **Use via OpenAI API** - Access everything through one interface
   ```python
   # Same client for both models!
   client = OpenAI(base_url="https://litellm.domain.com")
   client.chat.completions.create(model="gpt-oss", ...)
   client.embeddings.create(model="nomic-embed", ...)
   ```

---

**Prerequisites**: 
- ✅ Completed `00-platform-validation.ipynb`
- ✅ Chat model deployed (via `tkt-tensorrt-llm`)
- ✅ Embedding model deployed (via `tkt-text-embeddings`)

Let's get started!

In [7]:
import os
import httpx
from IPython.display import display, HTML

def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✓ {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">✗ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ {msg}</span>'))

In [8]:
DOMAIN_NAME = os.environ.get('DOMAIN_NAME')
print(DOMAIN_NAME)

cmxela.com


---
## 📋 Configuration: Know Your Endpoints

Before registering models, we need to know **where they're deployed**. Your models are running as Kubernetes services with HTTPS endpoints.

### 🔍 What Are These URLs?

- **`LITELLM_ENDPOINT`** - The LiteLLM gateway URL (e.g., `https://litellm.domain.com`)
- **`LLM_SERVICE_URL`** - Your chat model service (e.g., `https://gptoss20.domain.com`)
- **`EMBEDDING_SERVICE_URL`** - Your embedding model service (e.g., `https://nomic.domain.com`)

These URLs are exposed via Kubernetes Ingress and secured with TLS certificates.

### 🎯 Model Names

You'll register these models with **friendly names** that you'll use in your code:
- `gpt-oss` - Your chat completion model
- `nomic-embed` - Your embedding model

💡 **Tip**: These names are arbitrary! You could call them `my-llm` and `my-embeddings` if you prefer.

In [9]:
# LiteLLM configuration
LITELLM_ENDPOINT = os.environ.get('LITELLM_ENDPOINT')
LITELLM_MASTER_KEY = os.environ.get('LITELLM_MASTER_KEY')

# Your deployed services - UPDATE THESE to match your deployment
# Format: http://{service-name}.{namespace}.svc:{port}
LLM_SERVICE_URL = "https://gptoss20." + DOMAIN_NAME  # Your tkt-tensorrt-llm deployment
LLM_MODEL_NAME = "gpt-oss-20b"  # Name to register in LiteLLM

EMBEDDING_SERVICE_URL = "https://nomic." + DOMAIN_NAME  # Your tkt-text-embeddings deployment
EMBEDDING_MODEL_NAME = "nomic-embed"  # Name to register in LiteLLM

print(f"LiteLLM Endpoint: {LITELLM_ENDPOINT}")
print(f"LLM Service: {LLM_SERVICE_URL}")
print(f"Embedding Service: {EMBEDDING_SERVICE_URL}")

LiteLLM Endpoint: https://litellm.cmxela.com
LLM Service: https://gptoss20.cmxela.com
Embedding Service: https://nomic.cmxela.com


---
## 🏥 Step 1: Health Check Your Services

Before registering models in LiteLLM, let's verify that your deployed services are **healthy and reachable**.

### 🎯 What We're Testing

Each service exposes a `/health` endpoint that returns:
- ✅ **Status** - Is the service running?
- 🔬 **Model Info** - Which model is loaded?
- ⚙️ **Engine Details** - What backend is running (TensorRT-LLM, TEI, etc.)?

### 💡 Why This Matters

If these health checks fail, registration will also fail. Common issues:
- 🔴 Service not deployed yet
- 🔴 Wrong DNS/domain configuration
- 🔴 Model still loading (can take 2-5 minutes)
- 🔴 SSL certificate issues

> **Note**: We use `verify=False` for SSL because we're using self-signed certificates in the cluster.

In [10]:
# Check LLM service health
try:
    with httpx.Client(timeout=10.0, verify=False) as client:
        resp = client.get(f"{LLM_SERVICE_URL}/health")
        if resp.status_code == 200:
            success(f"LLM service healthy: {LLM_SERVICE_URL}")
            info(f"Response: {resp.json()}")
        else:
            error(f"LLM service returned {resp.status_code}")
except Exception as e:
    error(f"Cannot reach LLM service: {e}")
    info("Make sure your tkt-tensorrt-llm deployment is running")

In [5]:
# Check Embedding service health
try:
    with httpx.Client(timeout=10.0, verify=False) as client:
        resp = client.get(f"{EMBEDDING_SERVICE_URL}/health")
        if resp.status_code == 200:
            success(f"Embedding service healthy: {EMBEDDING_SERVICE_URL}")
            info(f"Response: {resp.json()}")
        else:
            error(f"Embedding service returned {resp.status_code}")
except Exception as e:
    error(f"Cannot reach Embedding service: {e}")
    info("Make sure your tkt-text-embeddings deployment is running")

---
## 🔧 Step 2: Register Chat Model in LiteLLM

Now let's register your **chat completion model** so it's accessible via the LiteLLM gateway.

### 🧠 What is a Chat Model?

Chat models generate **conversational responses**. They take a list of messages and return an AI-generated reply.

**Use cases in the Research Assistant**:
- 💬 Answering questions about papers
- ✍️ Summarizing research findings
- 🤖 Powering AI agents (researcher, analyst, writer)
- 🔍 Generating search queries

### 🎯 Registration Config Explained

```python
{
    "model_name": "gpt-oss",           # Your friendly name
    "litellm_params": {
        "model": "openai/gpt-oss",     # OpenAI-compatible format
        "api_base": "https://...",     # Your service URL
        "api_key": "not-needed",       # Local service, no auth
        "max_tokens": 8000             # ⚠️ CRITICAL: Default passed to backend
    },
    "model_info": {
        "description": "...",          # Human-readable description
        "mode": "chat",                # Type: chat or embedding
        "max_tokens": 131072,          # Context window size
        "max_output_tokens": 16384     # Max allowed output tokens
    }
}
```

### ⚠️ Critical: max_tokens in litellm_params

LiteLLM has a known issue where it doesn't always forward `max_tokens` from client requests to custom OpenAI-compatible backends. 

**The fix**: Set `max_tokens` directly in `litellm_params`. This value becomes the default for ALL requests to this model, ensuring your responses aren't truncated.

- `litellm_params.max_tokens: 8000` → Actually passed to the backend
- `model_info.max_output_tokens: 16384` → Metadata only (not passed through)

### 🔄 Update Support

This cell supports **updating** existing registrations by deleting and re-registering.

In [6]:
# Register LLM model (ALWAYS updates - deletes and re-registers)
# IMPORTANT: max_tokens in litellm_params gets passed to the backend
# IMPORTANT: use_in_pass_through enables passthrough for batch requests
# IMPORTANT: max_input_tokens sets the INPUT context limit (prevents 8192 default)
llm_config = {
    "model_name": LLM_MODEL_NAME,
    "litellm_params": {
        "model": f"openai/{LLM_MODEL_NAME}",
        "api_base": f"{LLM_SERVICE_URL}/v1",
        "api_key": "not-needed",  # Local service, no API key required
        "max_tokens": 8000,       # Default max_tokens passed to backend
        "use_in_pass_through": True  # Enable passthrough for batch requests
    },
    "model_info": {
        "description": "Local GPT-OSS 20B via TensorRT-LLM",
        "mode": "chat",
        "max_tokens": 131072,        # Model context window (128K)
        "max_input_tokens": 131072,  # INPUT context limit (prevents 8192 default!)
        "max_output_tokens": 16384   # Max allowed output tokens
    }
}

try:
    with httpx.Client(timeout=30.0, verify=False) as client:
        # First, check if model already exists and DELETE it to update config
        resp = client.get(
            f"{LITELLM_ENDPOINT}/model/info",
            headers={"Authorization": f"Bearer {LITELLM_MASTER_KEY}"}
        )
        
        if resp.status_code == 200:
            models = resp.json()
            for model in models.get('data', []):
                if model.get('model_name') == LLM_MODEL_NAME:
                    model_id = model.get('model_info', {}).get('id')
                    info(f"Model '{LLM_MODEL_NAME}' exists, deleting to update...")
                    delete_resp = client.post(
                        f"{LITELLM_ENDPOINT}/model/delete",
                        headers={
                            "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
                            "Content-Type": "application/json"
                        },
                        json={"id": model_id}
                    )
                    if delete_resp.status_code == 200:
                        info("Deleted old registration")
                    break
        
        # Register the model with updated config
        resp = client.post(
            f"{LITELLM_ENDPOINT}/model/new",
            headers={
                "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
                "Content-Type": "application/json"
            },
            json=llm_config
        )
        
        if resp.status_code == 200:
            success(f"Registered LLM model: {LLM_MODEL_NAME}")
            info(f"max_input_tokens: 131072 (128K context window)")
            info(f"max_output_tokens: 16384")
            info(f"Default max_tokens: 8000 (in litellm_params)")
        else:
            error(f"Failed to register LLM: {resp.status_code}")
            info(f"Response: {resp.text}")
except Exception as e:
    error(f"Error registering LLM: {e}")

---
## 🧮 Step 3: Register Embedding Model in LiteLLM

Now let's register your **embedding model** for vector search capabilities.

### 🎯 What are Embeddings?

Embeddings convert text into **high-dimensional vectors** (numbers) that capture semantic meaning. Similar concepts have similar vectors!

**Example**:
- "machine learning" → `[0.23, -0.45, 0.12, ...]` (768 dimensions)
- "deep learning" → `[0.25, -0.43, 0.14, ...]` ← Very similar!
- "cooking recipes" → `[-0.67, 0.89, -0.34, ...]` ← Very different!

### 💪 Use Cases in Research Assistant

- 🔍 **Semantic Search** - Find papers by meaning, not just keywords
- 📊 **Similarity Matching** - "Find papers similar to this one"
- 🎯 **RAG Retrieval** - Get relevant context for answering questions
- 📁 **Clustering** - Group related papers together

### 🔬 Your Model: nomic-embed-text-v1.5

- **Dimensions**: 768 (each text becomes a 768-number vector)
- **Max Length**: 8,192 tokens (very long context!)
- **Training**: Trained on diverse text (scientific papers, code, web)
- **Performance**: State-of-the-art for retrieval tasks

### 🔄 Idempotent Registration

Like the chat model, this is also **idempotent** - safe to run multiple times!

In [8]:
# Register Embedding model (idempotent)
embedding_config = {
    "model_name": EMBEDDING_MODEL_NAME,
    "litellm_params": {
        "model": f"openai/{EMBEDDING_MODEL_NAME}",
        "api_base": f"{EMBEDDING_SERVICE_URL}/v1",
        "api_key": "not-needed"
    },
    "model_info": {
        "description": "Local nomic-embed-text-v1.5 embeddings",
        "mode": "embedding"
    }
}

try:
    with httpx.Client(timeout=30.0, verify=False) as client:
        # First, check if model already exists
        resp = client.get(
            f"{LITELLM_ENDPOINT}/model/info",
            headers={"Authorization": f"Bearer {LITELLM_MASTER_KEY}"}
        )
        
        model_exists = False
        if resp.status_code == 200:
            models = resp.json()
            model_exists = any(
                model.get('model_name') == EMBEDDING_MODEL_NAME 
                for model in models.get('data', [])
            )
        
        if model_exists:
            info(f"Model '{EMBEDDING_MODEL_NAME}' already registered, skipping...")
        else:
            # Register the model
            resp = client.post(
                f"{LITELLM_ENDPOINT}/model/new",
                headers={
                    "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
                    "Content-Type": "application/json"
                },
                json=embedding_config
            )
            
            if resp.status_code == 200:
                success(f"Registered Embedding model: {EMBEDDING_MODEL_NAME}")
                info(f"Response: {resp.json()}")
            else:
                error(f"Failed to register Embedding: {resp.status_code}")
                info(f"Response: {resp.text}")
except Exception as e:
    error(f"Error registering Embedding: {e}")

---
## ✅ Step 4: Verify Registration

Let's confirm that both models are now registered in LiteLLM.

### 🔍 What We're Checking

LiteLLM maintains a **model registry** that lists all configured models. Each entry includes:
- 📛 **Model Name** - The friendly name you'll use in code
- 🎭 **Mode** - Type: `chat` (text generation) or `embedding` (vectors)
- 🔗 **API Base** - Where the actual model service is running
- 📋 **Metadata** - Description, configuration, timestamps

### 🎯 Expected Output

You should see at least these two models:
- ✅ `gpt-oss` (chat) - Your chat completion model
- ✅ `nomic-embed` (embedding) - Your embedding model

> **Note**: You might see other models too if they were registered previously!

In [9]:
# List registered models
try:
    with httpx.Client(timeout=30.0, verify=False) as client:
        resp = client.get(
            f"{LITELLM_ENDPOINT}/model/info",
            headers={"Authorization": f"Bearer {LITELLM_MASTER_KEY}"}
        )
        
        if resp.status_code == 200:
            models = resp.json()
            success(f"Found {len(models.get('data', []))} registered models")
            for model in models.get('data', []):
                model_name = model.get('model_name', 'unknown')
                mode = model.get('model_info', {}).get('mode', 'unknown')
                info(f"  - {model_name} ({mode})")
        else:
            error(f"Failed to list models: {resp.status_code}")
except Exception as e:
    error(f"Error listing models: {e}")

---
## 🧪 Step 5: End-to-End Testing

Now for the fun part - let's **actually use** your models through the LiteLLM gateway!

### 🎯 What We're Testing

We'll use the **OpenAI Python SDK** pointed at your LiteLLM endpoint. This proves that:
- ✅ Your models are properly registered
- ✅ LiteLLM can route requests correctly
- ✅ The underlying services are responding
- ✅ Authentication is working

### 🔌 The OpenAI Client

```python
from openai import OpenAI

client = OpenAI(
    base_url=LITELLM_ENDPOINT,    # Your LiteLLM gateway
    api_key=LITELLM_MASTER_KEY     # LiteLLM auth key
)
```

### 💡 Why This Works

LiteLLM provides an **OpenAI-compatible API**, which means:
- 🔄 Any tool that works with OpenAI works with your local models
- 📚 LangChain, CrewAI, LlamaIndex all work out-of-the-box
- 🛠️ No custom integration code needed
- 🔐 Single authentication mechanism

### 🧪 Test 1: Chat Completion

We'll ask the model a simple question to verify text generation works.

### 🧪 Test 2: Embeddings

We'll convert text to vectors to verify embedding generation works.

In [10]:
from openai import OpenAI

# Create client pointing to LiteLLM
client = OpenAI(
    base_url=LITELLM_ENDPOINT,
    api_key=LITELLM_MASTER_KEY
)

info(f"OpenAI client configured for LiteLLM at {LITELLM_ENDPOINT}")

In [11]:
# Test Chat Completion
try:
    response = client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[{"role": "user", "content": "Say 'Hello from Thinkube!' in exactly those words."}],
        max_tokens=20
    )
    
    success(f"Chat completion via LiteLLM works!")
    info(f"Model: {LLM_MODEL_NAME}")
    info(f"Response: {response.choices[0].message.content}")
except Exception as e:
    error(f"Chat completion failed: {e}")

In [12]:
# Test Embeddings
try:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        input="This is a test sentence for embedding."
    )
    
    embedding = response.data[0].embedding
    success(f"Embeddings via LiteLLM works!")
    info(f"Model: {EMBEDDING_MODEL_NAME}")
    info(f"Embedding dimensions: {len(embedding)}")
    info(f"First 5 values: {embedding[:5]}")
except Exception as e:
    error(f"Embeddings failed: {e}")

---
## 🎓 Summary: Your Unified AI Gateway is Ready!

Congratulations! You've successfully configured your LiteLLM gateway. Here's what you accomplished:

### ✅ What You Built

1. ✅ **Health-checked** your deployed services
2. ✅ **Registered** chat model (`gpt-oss`) in LiteLLM
3. ✅ **Registered** embedding model (`nomic-embed`) in LiteLLM
4. ✅ **Verified** both models appear in the registry
5. ✅ **Tested** end-to-end functionality

### 🎯 What This Enables

Now you have a **single API endpoint** that provides:

```python
from openai import OpenAI

client = OpenAI(
    base_url=os.environ['LITELLM_ENDPOINT'],
    api_key=os.environ['LITELLM_MASTER_KEY']
)

# Chat Completion - Generate text responses
response = client.chat.completions.create(
    model="gpt-oss",
    messages=[
        {"role": "user", "content": "Explain RAG"}
    ]
)

# Embeddings - Convert text to vectors
embeddings = client.embeddings.create(
    model="nomic-embed",
    input="Parameter-efficient fine-tuning with LoRA"
)
```

### 💪 Key Benefits

✅ **OpenAI-Compatible** - Works with LangChain, CrewAI, LlamaIndex  
✅ **Single Endpoint** - One API for all models  
✅ **Cost Tracking** - LiteLLM logs usage and costs  
✅ **Load Balancing** - Distribute requests across replicas  
✅ **Fallbacks** - Automatic retry with backup models  
✅ **Model Switching** - Change models without code changes  

### 🔄 The Complete Flow

```mermaid
graph LR
    A[🤖 Your App] --> B[🔤 LiteLLM<br/>litellm.domain.com]
    B --> C[💬 gpt-oss<br/>Chat Model]
    B --> D[🧮 nomic-embed<br/>Embeddings]
    C --> E[📈 Langfuse<br/>Observability]
    D --> E
    
    style A fill:#e1f5ff
    style B fill:#fff4e6
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fce4ec
```

### 🎯 Real-World Use Cases

Now that your gateway is configured, you can:

1. **📚 Build RAG Systems** - Embed documents, search semantically, generate answers
2. **🤖 Create AI Agents** - Multiple agents sharing the same model gateway
3. **🔍 Semantic Search** - Find similar content using embeddings
4. **💬 Chatbots** - Conversational interfaces for your data
5. **✍️ Content Generation** - Summarization, translation, writing assistance

### 🚀 Next Steps

**Continue to**: `research-assistant/02-langchain-rag.ipynb`

In the next notebook, you'll use these models to build a **complete RAG pipeline**:
- 📥 Fetch research papers from ArXiv
- ✂️ Chunk papers into searchable pieces
- 🧮 Generate embeddings with `nomic-embed`
- 💾 Store vectors in Qdrant
- 🔍 Perform semantic search
- 💬 Answer questions using `gpt-oss`

---

💡 **Remember**: All models run **locally on Thinkube** - no external API calls or costs!